# 300k + VAE Fidelity Benchmark

## Install requirements

In [ ]:
%pip install -q "numpy<2" pandas scikit-learn matplotlib cloudpickle interpret tensorflow==2.15.1 tensorflow-lattice gaminet

## Imports and paths

In [ ]:
import os
import pickle
import sys
import warnings
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import cloudpickle
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

ROOT = Path.cwd()
MODEL_DIR = ROOT / "model"
CSV_DIR = ROOT / "processed_csv"
PLOT_DIR = ROOT / "plots_300k_vae"
PREDICT_BATCH_SIZE = 50000

TEACHERS = ["random_forest", "deep_neural_net"]
SURROGATES = ["logistic_regression", "ebm", "gaminet"]

## Helper functions

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }


def batched_predict(model, X, batch_size=PREDICT_BATCH_SIZE):
    preds = []
    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))
        batch = X.iloc[start:stop] if hasattr(X, "iloc") else X[start:stop]
        preds.append(np.asarray(model.predict(batch)).astype(int))
    return np.concatenate(preds, axis=0)


def gaminet_predict_labels(model, X, batch_size=8192):
    preds = []
    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))
        batch_pred = np.asarray(model.predict(X[start:stop]), dtype=np.float32).reshape(-1)
        if np.nanmin(batch_pred) < 0.0 or np.nanmax(batch_pred) > 1.0:
            batch_pred = 1.0 / (1.0 + np.exp(-batch_pred))
        preds.append((batch_pred >= 0.5).astype(np.int32))
    return np.concatenate(preds, axis=0)


def load_cloudpickle(path):
    if "numpy._core.numeric" not in sys.modules:
        import numpy.core.numeric as numpy_core_numeric
        sys.modules["numpy._core.numeric"] = numpy_core_numeric
    with path.open("rb") as handle:
        return cloudpickle.load(handle)


def ensure_gaminet_runtime():
    import tensorflow as tf
    from gaminet import GAMINet
    try:
        tf.config.set_visible_devices([], "GPU")
    except Exception:
        pass
    try:
        tf.config.experimental.set_visible_devices([], "GPU")
    except Exception:
        pass
    return tf, GAMINet


def load_gaminet_bundle(model_path):
    _, GAMINet = ensure_gaminet_runtime()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with model_path.open("rb") as handle:
            model_dict = pickle.load(handle)
    model = GAMINet(
        meta_info=model_dict["meta_info"],
        subnet_arch=model_dict["subnet_arch"],
        interact_arch=model_dict["interact_arch"],
        lr_bp=model_dict["lr_bp"],
        batch_size=model_dict["batch_size"],
        task_type=model_dict["task_type"],
        activation_func=model_dict["activation_func"],
        tuning_epochs=model_dict["tuning_epochs"],
        main_effect_epochs=model_dict["main_effect_epochs"],
        interaction_epochs=model_dict["interaction_epochs"],
        early_stop_thres=model_dict["early_stop_thres"],
        heredity=model_dict["heredity"],
        reg_clarity=model_dict["reg_clarity"],
        loss_threshold=model_dict["loss_threshold"],
        mono_increasing_list=model_dict["mono_increasing_list"],
        mono_decreasing_list=model_dict["mono_decreasing_list"],
        lattice_size=model_dict["lattice_size"],
        verbose=False,
        val_ratio=model_dict["val_ratio"],
        random_state=model_dict["random_state"],
        interact_num=model_dict["interact_num"],
    )
    model.load(folder=str(model_path.parent) + "/", name=model_path.stem)
    return model, model_dict


def scale_with_saved_meta_info(X, meta_info):
    feature_names = [name for name, payload in meta_info.items() if payload.get("type") != "target"]
    X_np = X[feature_names].astype(np.float32, copy=False).to_numpy(dtype=np.float32, copy=True)
    X_scaled = np.zeros_like(X_np, dtype=np.float32)
    for idx, feature_name in enumerate(feature_names):
        scaler = meta_info[feature_name]["scaler"]
        X_scaled[:, [idx]] = scaler.transform(X_np[:, [idx]]).astype(np.float32)
    return X_scaled, feature_names


def error_report(teacher_pred, surrogate_pred, y_true):
    masks = {
        "overall_error": teacher_pred != y_true,
        "false_positive": (teacher_pred == 1) & (y_true == 0),
        "false_negative": (teacher_pred == 0) & (y_true == 1),
    }
    out = {}
    for name, mask in masks.items():
        out[f"{name}_count"] = int(mask.sum())
        out[f"{name}_fidelity"] = float(np.mean(surrogate_pred[mask] == teacher_pred[mask])) if mask.any() else np.nan
        out[f"{name}_truth_match"] = float(np.mean(surrogate_pred[mask] == y_true[mask])) if mask.any() else np.nan
    return out

## Load test data and teacher predictions

In [ ]:
processed = pd.read_csv(CSV_DIR / "processed_dataset_with_split.csv")
feature_cols = [col for col in processed.columns if col not in {"label", "split"}]
test_df = processed[processed["split"] == "test"].copy()
X_test = test_df[feature_cols].astype(np.float32)
y_test = test_df["label"].to_numpy(dtype=int, copy=True)
teacher_predictions = {
    "random_forest": np.load(MODEL_DIR / "teacher_real_test_predictions_random_forest.npy").astype(int),
    "deep_neural_net": np.load(MODEL_DIR / "teacher_real_test_predictions_deep_neural_net.npy").astype(int),
}
pd.DataFrame({"split": ["test"], "rows": [len(test_df)], "features": [len(feature_cols)]})

## Load surrogate models

In [ ]:
models = {
    "random_forest": {
        "logistic_regression": load_cloudpickle(MODEL_DIR / "logreg_mixed_300k_plus_vae_random_forest.pkl"),
        "ebm": load_cloudpickle(MODEL_DIR / "ebm_random_forest.pkl"),
    },
    "deep_neural_net": {
        "logistic_regression": load_cloudpickle(MODEL_DIR / "logreg_mixed_300k_plus_vae_deep_neural_net.pkl"),
        "ebm": load_cloudpickle(MODEL_DIR / "ebm_deep_neural_net.pkl"),
    },
}
gaminet_bundles = {
    "random_forest": load_gaminet_bundle(MODEL_DIR / "gaminet_random_forest.pickle"),
    "deep_neural_net": load_gaminet_bundle(MODEL_DIR / "gaminet_deep_neural_net.pickle"),
}
models["random_forest"]["gaminet"] = gaminet_bundles["random_forest"][0]
models["deep_neural_net"]["gaminet"] = gaminet_bundles["deep_neural_net"][0]
list(models.keys())

## Run surrogate inference

In [ ]:
predictions = {teacher: {} for teacher in TEACHERS}
for teacher in TEACHERS:
    predictions[teacher]["logistic_regression"] = batched_predict(models[teacher]["logistic_regression"], X_test)
    predictions[teacher]["ebm"] = batched_predict(models[teacher]["ebm"], X_test)
    gaminet_model, gaminet_payload = gaminet_bundles[teacher]
    X_gaminet, gaminet_feature_names = scale_with_saved_meta_info(X_test, gaminet_payload["meta_info"])
    predictions[teacher]["gaminet"] = gaminet_predict_labels(gaminet_model, X_gaminet)
pd.DataFrame({"teacher": TEACHERS, "surrogates": [list(predictions[t].keys()) for t in TEACHERS]})

## Overall fidelity and F1-fidelity

In [ ]:
rows = []
for teacher in TEACHERS:
    teacher_pred = teacher_predictions[teacher]
    for surrogate in SURROGATES:
        pred = predictions[teacher][surrogate]
        fidelity = compute_metrics(teacher_pred, pred)
        truth = compute_metrics(y_test, pred)
        rows.append({
            "teacher": teacher,
            "surrogate": surrogate,
            "test_accuracy_vs_truth": truth["accuracy"],
            "overall_fidelity_accuracy": fidelity["accuracy"],
            "f1_fidelity": fidelity["f1"],
            "fidelity_precision": fidelity["precision"],
            "fidelity_recall": fidelity["recall"],
        })
overall_fidelity_df = pd.DataFrame(rows)
overall_fidelity_df

## Error fidelity on teacher mistakes

In [ ]:
rows = []
for teacher in TEACHERS:
    teacher_pred = teacher_predictions[teacher]
    for surrogate in SURROGATES:
        pred = predictions[teacher][surrogate]
        report = error_report(teacher_pred, pred, y_test)
        rows.append({
            "teacher": teacher,
            "surrogate": surrogate,
            "teacher_error_count": report["overall_error_count"],
            "error_fidelity": report["overall_error_fidelity"],
            "matches_truth_on_teacher_errors": report["overall_error_truth_match"],
        })
error_fidelity_df = pd.DataFrame(rows)
error_fidelity_df

## FP and FN error fidelity

In [ ]:
rows = []
for teacher in TEACHERS:
    teacher_pred = teacher_predictions[teacher]
    for surrogate in SURROGATES:
        pred = predictions[teacher][surrogate]
        report = error_report(teacher_pred, pred, y_test)
        rows.append({
            "teacher": teacher,
            "surrogate": surrogate,
            "fp_count": report["false_positive_count"],
            "fp_fidelity": report["false_positive_fidelity"],
            "fn_count": report["false_negative_count"],
            "fn_fidelity": report["false_negative_fidelity"],
        })
detailed_error_fidelity_df = pd.DataFrame(rows)
detailed_error_fidelity_df

## Baseline comparison table

In [ ]:
saved_comparison = pd.read_csv(CSV_DIR / "logreg_vs_ebm_vs_gaminet_comparison.csv")
saved_comparison[saved_comparison["variant"] == "mixed_300k_plus_vae"]

## Save notebook-computed tables

In [ ]:
overall_fidelity_df.to_csv(CSV_DIR / "notebook_overall_fidelity_300k_vae.csv", index=False)
error_fidelity_df.to_csv(CSV_DIR / "notebook_error_fidelity_300k_vae.csv", index=False)
detailed_error_fidelity_df.to_csv(CSV_DIR / "notebook_detailed_error_fidelity_300k_vae.csv", index=False)
pd.DataFrame({"saved": [
    str(CSV_DIR / "notebook_overall_fidelity_300k_vae.csv"),
    str(CSV_DIR / "notebook_error_fidelity_300k_vae.csv"),
    str(CSV_DIR / "notebook_detailed_error_fidelity_300k_vae.csv"),
]})

## Feature importance plots

In [ ]:
for image_name in [
    "ebm_rf_feature_importance.png",
    "ebm_dnn_feature_importance.png",
    "gaminet_rf_feature_importance.png",
    "gaminet_dnn_feature_importance.png",
]:
    display(Image(filename=str(PLOT_DIR / image_name), width=900))

## EBM vs GAMI-Net interpretation panels

In [ ]:
display(Image(filename=str(PLOT_DIR / "rf_ebm_vs_gaminet_common_top5_main_effects.png"), width=1100))
display(Image(filename=str(PLOT_DIR / "dnn_ebm_vs_gaminet_common_top5_main_effects.png"), width=1100))